## import libraries

In [9]:
import pystac_client
import planetary_computer
import rasterio
from pyproj import Transformer
import pandas as pd

## set up the catalog

In [10]:
catalog = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1", 
                                    modifier=planetary_computer.sign_inplace)

In [11]:
def get_point_data(lat, lon, year):
    point_geom = {"type": "Point", "coordinates": [lon, lat]}
    time_range = f"{year}-01-01/{year}-12-31"
    
    # --- Part A: Sentinel-2 (Optical) ---
    s2_search = catalog.search(
        collections=["sentinel-2-l2a"],
        intersects=point_geom,
        datetime=time_range,
        query={"eo:cloud_cover": {"lt": 20}}
    )
    
    # --- Part B: Sentinel-1 (Radar) ---
    s1_search = catalog.search(
        collections=["sentinel-1-grd"],
        intersects=point_geom,
        datetime=time_range,
        query={"sar:instrument_mode": {"eq": "IW"}}
    )

    def fetch_pixel(item, bands, lon, lat):
        results = {"date": item.datetime.date()}
        for band in bands:
            try:
                with rasterio.open(item.assets[band].href) as src:
                    transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
                    px, py = transformer.transform(lon, lat)
                    val = list(src.sample([(px, py)]))[0][0]
                    results[band] = val
            except Exception:
                results[band] = None
        return results

    # Execute Extraction
    s2_data = [fetch_pixel(it, ["B04", "B08"], lon, lat) for it in s2_search.get_items()]
    s1_data = [fetch_pixel(it, ["vv", "vh"], lon, lat) for it in s1_search.get_items()]

    return pd.DataFrame(s2_data), pd.DataFrame(s1_data)

# Example usage for your thesis dataset
# lat, lon = ...
# df_s2, df_s1 = get_point_data(lat, lon, 2023)

In [12]:
lon, lat = 14.361975914287918, 48.18615310836688
df_s2, df_s1 = get_point_data(lat, lon, 2019)

/Users/devseed/Documents/docs/THESIS/SpatialTemporal_generalization_of_SITS_FM/.venv/lib/python3.12/site-packages/pystac_client/item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(
